# SLM Fine-Tuning on Google Colab (Unsloth QLoRA)

Train the local **SLM Finetuning** project on a **Colab GPU** (CUDA), then download the adapter + metrics.

**Runtime → Change runtime type → GPU** (T4 / L4 / A100).

| Environment | Backend |
|-------------|--------|
| Mac (local) | Unsloth MLX |
| Colab (this notebook) | Unsloth CUDA + TRL |

## 1) Check GPU

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "Enable a GPU runtime: Runtime → Change runtime type → GPU"
print("CUDA:", torch.cuda.get_device_name(0))

## 2) Get the project onto Colab

Pick **one** option below.

### Option A — Clone from GitHub
Set `REPO_URL` to your remote (public or with token).

In [ ]:
import os
from pathlib import Path

# >>> EDIT THIS <<<
REPO_URL = ""  # e.g. https://github.com/<you>/slm-finetuning.git
BRANCH = "main"
PROJECT_DIR = Path("/content/SLM-Finetuning")

if REPO_URL:
    if PROJECT_DIR.exists():
        %cd {PROJECT_DIR}
        !git pull
    else:
        !git clone -b {BRANCH} {REPO_URL} {PROJECT_DIR}
        %cd {PROJECT_DIR}
else:
    print("REPO_URL empty — use Option B (Drive) or Option C (upload zip).")

### Option B — Google Drive folder
Upload the project to Drive, then set `DRIVE_PROJECT_PATH`.

In [ ]:
from pathlib import Path

# >>> EDIT THIS <<<
USE_DRIVE = False
DRIVE_PROJECT_PATH = "/content/drive/MyDrive/SLM Finetuning"

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path(DRIVE_PROJECT_PATH)
    assert PROJECT_DIR.exists(), f"Not found: {PROJECT_DIR}"
    %cd {PROJECT_DIR}
    print("Using Drive project:", PROJECT_DIR)
else:
    print("Drive option skipped.")

### Option C — Upload a zip
Zip the project locally (exclude `.venv`), upload, then extract.

In [ ]:
from pathlib import Path

USE_ZIP_UPLOAD = False
ZIP_NAME = "SLM-Finetuning.zip"
PROJECT_DIR = Path("/content/SLM-Finetuning")

if USE_ZIP_UPLOAD:
    from google.colab import files
    import zipfile
    uploaded = files.upload()  # choose your zip
    zpath = Path(list(uploaded.keys())[0])
    PROJECT_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zpath, "r") as zf:
        zf.extractall(PROJECT_DIR)
    # if zip contains a single top folder, enter it
    kids = [p for p in PROJECT_DIR.iterdir() if p.is_dir() and (p / "pyproject.toml").exists()]
    if kids:
        PROJECT_DIR = kids[0]
    %cd {PROJECT_DIR}
    print("Extracted to", PROJECT_DIR)
else:
    print("Zip upload skipped.")

## 3) Install dependencies (Unsloth CUDA)

In [ ]:
import os
from pathlib import Path

# Ensure we are inside the project
if Path("pyproject.toml").exists():
    PROJECT_DIR = Path(".").resolve()
elif "PROJECT_DIR" in globals():
    %cd {PROJECT_DIR}
else:
    raise SystemExit("Project not found. Run Option A/B/C first.")

print("Project:", Path(".").resolve())

# Unsloth recommended Colab install
!pip install -q --upgrade pip
!pip install -q unsloth
!pip install -q -e .

print("Install done.")

## 4) (Optional) Generate / refresh credit-card dataset

In [ ]:
REGENERATE_DATA = False  # set True to rebuild 8000 train + 800 eval rows

if REGENERATE_DATA:
    !python scripts/generate_credit_card_dataset.py

!wc -l data/processed/train.jsonl data/processed/eval.jsonl

## 5) Train with Colab configs (QLoRA)

In [ ]:
import os
os.environ.setdefault("WANDB_DISABLED", "true")

!slm train \
  --method qlora \
  --model-config configs/model/colab.yaml \
  --training-config configs/training/colab.yaml \
  --run-name card-ops-colab

## 6) Metrics

In [ ]:
!slm metrics list-runs
!slm metrics latest

# Demo comparison table shape (no second model load):
!slm metrics report --demo

### Base vs fine-tuned report (loads both models — needs GPU RAM)

Uses the **held-out** eval set (`data/evaluation/heldout_comparison_eval.jsonl`) — questions **not** used in fine-tuning. Prints metrics plus each query with base vs fine-tuned answers.

After training finishes, set `RUN_ID` to your adapter folder name under `models/finetuned/`.

In [ ]:
from pathlib import Path

ft_dirs = sorted(Path("models/finetuned").glob("*"))
ft_dirs = [p for p in ft_dirs if p.is_dir() and p.name != "README.md"]
print("Available adapters:")
for p in ft_dirs:
    print(" -", p.name)

RUN_ID = "card-ops-colab-2"  # change if needed
adapter = Path("models/finetuned") / RUN_ID
if adapter.exists():
    !slm metrics report \
      --finetuned-model-dir {adapter} \
      --model-config configs/model/colab.yaml \
      --eval-file data/evaluation/heldout_comparison_eval.jsonl
else:
    print(f"Adapter not found yet: {adapter}")

## 7) Download artifacts to your laptop

In [ ]:
from pathlib import Path
import shutil
from google.colab import files

RUN_ID = "card-ops-colab"
bundle = Path("/content/slm_finetune_artifacts")
if bundle.exists():
    shutil.rmtree(bundle)
bundle.mkdir(parents=True)

for src in [
    Path("models/finetuned") / RUN_ID,
    Path("experiments") / RUN_ID,
    Path("artifacts/reports") / RUN_ID,
    Path("artifacts/metrics/metrics.db"),
]:
    if not src.exists():
        print("skip missing", src)
        continue
    dest = bundle / src.name
    if src.is_dir():
        shutil.copytree(src, dest)
    else:
        shutil.copy2(src, dest)

zip_path = shutil.make_archive(str(bundle), "zip", root_dir=bundle)
print("Created", zip_path)
files.download(zip_path)

## Notes

- Local Mac uses `configs/training/default.yaml` (MLX).
- Colab uses `configs/training/colab.yaml` + `configs/model/colab.yaml` (CUDA).
- Free Colab sessions can disconnect; save adapters to Drive if runs are long.
- After download, copy the adapter into your laptop `models/finetuned/<run_id>/`.